<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 02 · 项目规则变了，记忆怎么办

上线前，团队把批量导入上限从 100 行改成了 200 行。旧约定曾经是对的，如今却可能误导后续工作。
我们希望新的会话使用新规则，同时仍能回答“当时依据的是哪个版本”。

这一篇从一次修订开始，接着模拟两个人同时修改，最后让一条过期约定退出日常召回。

**这一篇的收获：** 修订和停用 Memory，读取历史版本，理解为什么更新时要带上自己检查过的 citation。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇不需要模型或 API Key。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("02", features=())
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 02",
        summary="第 02 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-02",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 建立最初的约定

本篇会重新创建自己的数据，不需要上一篇的运行状态。先保存 100 行的上限，并把这次的引用命名为 `original_citation`。


这里维护的是日常 Memory 中的一条知识。`entry_id` 标识条目；`memory_ref` 标识包含它的整份制品版本。统一制品读取与条目修改可以配合使用。

In [ ]:
from powercontext.http import (
    GetMemoryEntryRequest,
    ListArtifactsRequest,
    ListMemoryChangesRequest,
    ListMemoryEntriesRequest,
    RememberMemoryRequest,
    RetireMemoryEntryRequest,
    ReviseMemoryEntryRequest,
    SearchMemoryRequest,
)

original = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="constraint",
        text="batch_limit: 每批最多导入 100 行。",
        reason="首期导入约定",
    )
)
assert original.entry is not None
original_citation = original.entry.citation
show(original.entry)
memory_page = await client.list_artifacts(scope_id, "memory", ListArtifactsRequest())
assert any(item.artifact_id == original_citation.memory_ref.artifact_id for item in memory_page.items)
table([{"制品": item.artifact_id, "当前版本": item.revision} for item in memory_page.items])

## 2. 根据读到的版本进行修订

把 citation 和新内容一起交给 Server。它会检查：你读到的是否仍是可修改的当前版本。
成功后，条目的身份保持连续，内容版本向前推进。


In [ ]:
revised = await client.revise_memory_entry(
    ReviseMemoryEntryRequest(
        scope_id=scope_id,
        citation=original_citation,
        kind="constraint",
        text="batch_limit: 每批最多导入 200 行；超过时提示拆分文件。",
        reason="压测后确认新的上限",
    )
)
assert revised.entry is not None
latest_citation = revised.entry.citation
table([
    {"版本": "修改前", "条目": original_citation.entry_id, "内容": original.entry.text},
    {"版本": "修改后", "条目": latest_citation.entry_id, "内容": revised.entry.text},
])
assert latest_citation.entry_id == original_citation.entry_id
assert latest_citation.entry_version_id != original_citation.entry_version_id

## 3. 同事还拿着旧版本，会发生什么

假设另一位同事还没看到 200 行的更新，想把上限改为 150 行。下面故意带上旧 citation。
冲突是这一步期望看到的结果：它提醒调用方重新读取并判断，而不是悄悄覆盖别人的决定。


In [ ]:
from powercontext.client import ServerResponseError

try:
    await client.revise_memory_entry(
        ReviseMemoryEntryRequest(
            scope_id=scope_id,
            citation=original_citation,
            kind="constraint",
            text="batch_limit: 每批最多导入 150 行。",
            reason="基于旧版本的修改尝试",
        )
    )
except ServerResponseError as error:
    assert error.status_code == 409
    show({"HTTP": error.status_code, "结果": "旧版本修改被拒绝；先读取当前条目，再决定是否更新。"})
else:
    raise AssertionError("预期收到版本冲突。")

## 4. 当前召回和历史读取各自回答什么

搜索回答“现在可用的约定是什么”。精确读取回答“这个引用当时记录了什么”。
我们同时执行两种读取，观察它们并不矛盾。


In [ ]:
old_snapshot = await client.get_memory_entry(GetMemoryEntryRequest(scope_id=scope_id, citation=original_citation))
current = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="batch_limit", mode="fts"))
assert old_snapshot.text == original.entry.text
assert any(hit.text == revised.entry.text for hit in current.hits)
assert all(hit.text != original.entry.text for hit in current.hits)
table([
    {"读取方式": "精确历史引用", "内容": old_snapshot.text},
    *[{"读取方式": "当前搜索", "内容": hit.text} for hit in current.hits],
])
historical_artifact = await client.get_artifact_revision(
    scope_id,
    "memory",
    original_citation.memory_ref.artifact_id,
    original_citation.memory_ref.revision,
)
current_artifact = await client.get_artifact(scope_id, "memory", latest_citation.memory_ref.artifact_id)
assert current_artifact is not None and current_artifact.revision > historical_artifact.revision
show({"整份 Memory 的历史版本": historical_artifact.revision, "整份 Memory 的当前版本": current_artifact.revision})

## 5. 当规则不再适用时，让它退出召回

后来系统采用流式导入，整批行数上限已经失去意义。`retire_memory_entry` 会停用这条记忆。
停用会保留历史，所以不要把它理解成物理删除。


In [ ]:
retired = await client.retire_memory_entry(
    RetireMemoryEntryRequest(
        scope_id=scope_id,
        citation=latest_citation,
        reason="流式导入替代整批限制",
    )
)
active = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
history = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id, include_inactive=True))
assert all(entry.citation.entry_id != latest_citation.entry_id for entry in active.entries)
assert any(entry.state == "inactive" for entry in history.entries)
table([{"状态": entry.state, "内容": entry.text, "版本": entry.version} for entry in history.entries])

## 轮到你：读懂这次变化的理由

下面读取变更记录。先猜一猜会出现哪些操作，再运行查看 `op` 和 `reason`。
练习：新增一条 `line_number` 约束，把它修订一次，然后重新运行这个单元格观察新增的历史。
上面的写入与修订单元格就是可复用的答案骨架。


In [ ]:
changes = await client.list_memory_changes(ListMemoryChangesRequest(scope_id=scope_id, since_revision=0))
table([
    {"Revision": revision.memory_ref.revision, "操作": change.op, "理由": change.reason}
    for revision in changes.revisions
    for change in revision.changes
])
assert len(changes.revisions) >= 3

## 带着结果离开

你已经把“当前适用的知识”和“曾经记录的历史”区分开了。修订需要精确引用，停用影响 active recall，旧版本仍能用于追溯。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[同时做两个项目，如何避免串台](03_scopes.ipynb)。


In [ ]:
await lab.close()
print("本篇 Server 已关闭。")